## 1. Environment Setup

Install Required Library

In [ ]:
!pip install pyspark

#### 1.1 Create SparkContext and SparkSession

Remeber to create a session, which is an entry point to Spark

In [ ]:
# create entry points to spark
from pyspark.sql import SparkSession

ss  = SparkSession.builder \
                            .master("local[1]")\
                            .appName("SparkByExamples.com")\
                            .getOrCreate()
spark = ss.sparkContext

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("example").getOrCreate()
sc = spark.sparkContext

rdd = sc.parallelize([('a', 1), ('b', 2), ('c', 3)])
df = spark.createDataFrame(rdd, ['name', 'id'])
df.show()

+----+---+
|name| id|
+----+---+
|   a|  1|
|   b|  2|
|   c|  3|
+----+---+



## 2. From file/data to Dataframe

### 2.1 Create DataFrame by reading a file

In [ ]:
mtcars = ss.read.csv(path='mtcars.csv',
                        sep=',',
                        encoding='UTF-8',
                        comment=None,
                        header=True,
                        inferSchema=True)
mtcars.show(n=5, truncate=False)

+-----------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|_c0              |mpg |cyl|disp |hp |drat|wt   |qsec |vs |am |gear|carb|
+-----------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|Mazda RX4        |21.0|6  |160.0|110|3.9 |2.62 |16.46|0  |1  |4   |4   |
|Mazda RX4 Wag    |21.0|6  |160.0|110|3.9 |2.875|17.02|0  |1  |4   |4   |
|Datsun 710       |22.8|4  |108.0|93 |3.85|2.32 |18.61|1  |1  |4   |1   |
|Hornet 4 Drive   |21.4|6  |258.0|110|3.08|3.215|19.44|1  |0  |3   |1   |
|Hornet Sportabout|18.7|8  |360.0|175|3.15|3.44 |17.02|0  |0  |3   |2   |
+-----------------+----+---+-----+---+----+-----+-----+---+---+----+----+
only showing top 5 rows



### 2.2 Create DataFrame with hand-on data (e.g., from a list)

In [ ]:
my_list = [['a', 1], ['b', 2]]
df = ss.createDataFrame(my_list, ['letter', 'number']) # convert to Dataframe for processing
df.show()

+------+------+
|letter|number|
+------+------+
|     a|     1|
|     b|     2|
+------+------+



### 2.3 Create Pyspark Dataframe from Panda Dataframe

In [ ]:
import pandas as pd

# a Panda Dataframe
pdf = pd.DataFrame({
    'x': [[1,2,3], [4,5,6]],
    'y': [['a','b','c'], ['e','f','g']]
})

df_spark = ss.createDataFrame(pdf)
df_spark.show()

+---------+---------+
|        x|        y|
+---------+---------+
|[1, 2, 3]|[a, b, c]|
|[4, 5, 6]|[e, f, g]|
+---------+---------+



### 2.4 Datatype
Using `dtype`, you can view the Datatype of each column in the dataframe.
`Data Type` is a tuple of (Column Names, Column Datatype)

In [ ]:
df.dtypes

[('letter', 'string'), ('number', 'bigint')]

### 2.5 Create new Dataframe from exisiting Dataframe
Using ``toDF()``, we can create a new Dataframe from existing dataframe

In [ ]:
mtcars.select('mpg', 'cyl').toDF('mpg_1', 'cyl_1').show(5)

+-----+-----+
|mpg_1|cyl_1|
+-----+-----+
| 21.0|    6|
| 21.0|    6|
| 22.8|    4|
| 21.4|    6|
| 18.7|    8|
+-----+-----+
only showing top 5 rows



## 3. DataFrame Analytics

In Dataframe, data are stored in **column**.
There are different `methods` that can be used to edit the columns in the dataframe, for examples:
* `withColumnRenamed(existing, new)`: takes column names as arguments.
* `sort(*cols, **kwargs)`: column name (string) or column expression or **both**.
* `drop(*cols)`: ***remove a list of column names OR a single column expression.***
* `filter(condition)`: ***condition** refers to a column expression that returns `types.BooleanType` of values.
* `select(column)`: select a column by its name
* `printSchema()`: print the structure of a Dataframe

### 3.1 ``.columns``
``columns``: displays the column name of a Dataframe

In [ ]:
mtcars = ss.read.csv(path='mtcars.csv',
                        sep=',',
                        encoding='UTF-8',
                        comment=None,
                        header=True,
                        inferSchema=True)
mtcars.columns

['_c0',
 'mpg',
 'cyl',
 'disp',
 'hp',
 'drat',
 'wt',
 'qsec',
 'vs',
 'am',
 'gear',
 'carb']

### 3.2 ``.renames()``
As we can see in 3.1, the *index* column (i.e., c_0) does not have a name, so we rename it.

In [ ]:
df0 = mtcars.withColumnRenamed('_c0', 'model')
df0.show(5)

+-----------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|            model| mpg|cyl| disp| hp|drat|   wt| qsec| vs| am|gear|carb|
+-----------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|        Mazda RX4|21.0|  6|160.0|110| 3.9| 2.62|16.46|  0|  1|   4|   4|
|    Mazda RX4 Wag|21.0|  6|160.0|110| 3.9|2.875|17.02|  0|  1|   4|   4|
|       Datsun 710|22.8|  4|108.0| 93|3.85| 2.32|18.61|  1|  1|   4|   1|
|   Hornet 4 Drive|21.4|  6|258.0|110|3.08|3.215|19.44|  1|  0|   3|   1|
|Hornet Sportabout|18.7|  8|360.0|175|3.15| 3.44|17.02|  0|  0|   3|   2|
+-----------------+----+---+-----+---+----+-----+-----+---+---+----+----+
only showing top 5 rows



### 3.3 ``.sort()``
We can use ``sort()`` to sort a particular column

In [ ]:
# sorted the brand column
df0 = df0.sort('model')
df0.show(5)

+------------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|             model| mpg|cyl| disp| hp|drat|   wt| qsec| vs| am|gear|carb|
+------------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|       AMC Javelin|15.2|  8|304.0|150|3.15|3.435| 17.3|  0|  0|   3|   2|
|Cadillac Fleetwood|10.4|  8|472.0|205|2.93| 5.25|17.98|  0|  0|   3|   4|
|        Camaro Z28|13.3|  8|350.0|245|3.73| 3.84|15.41|  0|  0|   3|   4|
| Chrysler Imperial|14.7|  8|440.0|230|3.23|5.345|17.42|  0|  0|   3|   4|
|        Datsun 710|22.8|  4|108.0| 93|3.85| 2.32|18.61|  1|  1|   4|   1|
+------------------+----+---+-----+---+----+-----+-----+---+---+----+----+
only showing top 5 rows



### 3.4 ``.drop()``
Remove **a list** of column names OR **a single** column expression.

In [ ]:
df1 = df0.drop('mpg')
df1.show(5)

+------------------+---+-----+---+----+-----+-----+---+---+----+----+
|             model|cyl| disp| hp|drat|   wt| qsec| vs| am|gear|carb|
+------------------+---+-----+---+----+-----+-----+---+---+----+----+
|       AMC Javelin|  8|304.0|150|3.15|3.435| 17.3|  0|  0|   3|   2|
|Cadillac Fleetwood|  8|472.0|205|2.93| 5.25|17.98|  0|  0|   3|   4|
|        Camaro Z28|  8|350.0|245|3.73| 3.84|15.41|  0|  0|   3|   4|
| Chrysler Imperial|  8|440.0|230|3.23|5.345|17.42|  0|  0|   3|   4|
|        Datsun 710|  4|108.0| 93|3.85| 2.32|18.61|  1|  1|   4|   1|
+------------------+---+-----+---+----+-----+-----+---+---+----+----+
only showing top 5 rows



### 3.5 ``.filter()``
Filter a column based on a pre-defined **condition**

In [ ]:
df1 = df0.filter(mtcars.mpg > 18)
df1.show(5)

+------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|       model| mpg|cyl| disp| hp|drat|   wt| qsec| vs| am|gear|carb|
+------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|  Datsun 710|22.8|  4|108.0| 93|3.85| 2.32|18.61|  1|  1|   4|   1|
|Ferrari Dino|19.7|  6|145.0|175|3.62| 2.77| 15.5|  0|  1|   5|   6|
|    Fiat 128|32.4|  4| 78.7| 66|4.08|  2.2|19.47|  1|  1|   4|   1|
|   Fiat X1-9|27.3|  4| 79.0| 66|4.08|1.935| 18.9|  1|  1|   4|   1|
| Honda Civic|30.4|  4| 75.7| 52|4.93|1.615|18.52|  1|  1|   4|   2|
+------------+----+---+-----+---+----+-----+-----+---+---+----+----+
only showing top 5 rows



### 3.6 ``select()``
Select a column based on its **name**

In [ ]:
mtcars.select('mpg').show(5)

+----+
| mpg|
+----+
|21.0|
|21.0|
|22.8|
|21.4|
|18.7|
+----+
only showing top 5 rows



### 3.7 ``printSchema()``
``printSchema()`` can print the structure of the Dataframe

In [ ]:
df0.printSchema()

root
 |-- model: string (nullable = true)
 |-- mpg: double (nullable = true)
 |-- cyl: integer (nullable = true)
 |-- disp: double (nullable = true)
 |-- hp: integer (nullable = true)
 |-- drat: double (nullable = true)
 |-- wt: double (nullable = true)
 |-- qsec: double (nullable = true)
 |-- vs: integer (nullable = true)
 |-- am: integer (nullable = true)
 |-- gear: integer (nullable = true)
 |-- carb: integer (nullable = true)



### 3.8 ``distinct()``
We can use ``distinct()`` to display **unique** item in a column.

In [ ]:
mtcars.select('mpg').distinct().show(5)

+----+
| mpg|
+----+
|15.5|
|17.3|
|13.3|
|19.7|
|21.4|
+----+
only showing top 5 rows



### 3.9 Aggregate functions in Dataframe
PySpark provides many built-in standard Aggregate functions (e.g., sum, mean), these come in handy when we need to make aggregate operations on DataFrame columns.
Aggregate functions operate on a group of rows and calculate a single return value for every group.
For example,
* ``max``
* ``min``
* ``sum``
* ``mean``

In [ ]:
mtcars = ss.read.csv(path='mtcars.csv',
                        sep=',',
                        encoding='UTF-8',
                        comment=None,
                        header=True,
                        inferSchema=True)
mtcars.agg({'mpg': 'max'}).show()
mtcars.agg({'mpg': 'min'}).show()
mtcars.agg({'mpg': 'sum'}).show()
mtcars.agg({'mpg': 'mean'}).show()

+--------+
|max(mpg)|
+--------+
|    33.9|
+--------+

+--------+
|min(mpg)|
+--------+
|    10.4|
+--------+

+-----------------+
|         sum(mpg)|
+-----------------+
|642.8999999999999|
+-----------------+

+------------------+
|          avg(mpg)|
+------------------+
|20.090624999999996|
+------------------+



## 4 Dataframe to RDD

Here, we introduce how to convert dataframe to a RDD, so that the data can run in distributed machine.
This can easily be done by calling the `pyspark.sql.DataFrame.rdd()`.
Each element in the returned RDD is an `pyspark.sql.Row object`.
An Row is a list of **key-value** pairs.

In [ ]:
mtcars = ss.read.csv(path='mtcars.csv',
                        sep=',',
                        encoding='UTF-8',
                        comment=None,
                        header=True,
                        inferSchema=True)
mtcarsRDD = mtcars.rdd
mtcarsRDD.take(5)

[Row(_c0='Mazda RX4', mpg=21.0, cyl=6, disp=160.0, hp=110, drat=3.9, wt=2.62, qsec=16.46, vs=0, am=1, gear=4, carb=4),
 Row(_c0='Mazda RX4 Wag', mpg=21.0, cyl=6, disp=160.0, hp=110, drat=3.9, wt=2.875, qsec=17.02, vs=0, am=1, gear=4, carb=4),
 Row(_c0='Datsun 710', mpg=22.8, cyl=4, disp=108.0, hp=93, drat=3.85, wt=2.32, qsec=18.61, vs=1, am=1, gear=4, carb=1),
 Row(_c0='Hornet 4 Drive', mpg=21.4, cyl=6, disp=258.0, hp=110, drat=3.08, wt=3.215, qsec=19.44, vs=1, am=0, gear=3, carb=1),
 Row(_c0='Hornet Sportabout', mpg=18.7, cyl=8, disp=360.0, hp=175, drat=3.15, wt=3.44, qsec=17.02, vs=0, am=0, gear=3, carb=2)]

### 4.1 RDD example: ``map()`` and ``mapValue()``
With an RDD object, we can apply a set of mapping functions, such as ``map()`` and ``mapValue()``.

Recall:

The `map()` method applies a function to each elements of the RDD. Each element has to be a valid input to the function. The returned RDD has the function outputs as its new elements.

The `mapValues` function requires that each element in the RDD has a **key/value** pair structure, for example, a tuple of 2 items, or a list of 2 items. The `mapValues` function applies a function to each of the element values. The element key will remain unchanged.

In [ ]:
# retrieve the `c0` and `mpg` columns from mtcars (c0: car model).
mtcars_map = mtcarsRDD.map(lambda x: (x['_c0'], x['mpg']))
print('Map: \n', mtcars_map.take(5), '\n')

# multiply each value in `mpg' by 10
mtcars_mapvalues = mtcarsRDD.mapValues(lambda x: [x, x * 10])
print('MapValue: \n', mtcars_mapvalues.take(5))


Map: 
 [('Mazda RX4', 21.0), ('Mazda RX4 Wag', 21.0), ('Datsun 710', 22.8), ('Hornet 4 Drive', 21.4), ('Hornet Sportabout', 18.7)] 

MapValue: 
 [('Mazda RX4', [21.0, 210.0]), ('Mazda RX4 Wag', [21.0, 210.0]), ('Datsun 710', [22.8, 228.0]), ('Hornet 4 Drive', [21.4, 214.0]), ('Hornet Sportabout', [18.7, 187.0])]


### 4.2 RDD example: ``flatMapValues()``
With an RDD object, we can apply a set of mapping functions, such as ``flatMapValues()``.


The `flatMapValues()` function requires that each element in the RDD has a **key/value** pair structure, for example, a tuple of 2 items, or a list of 2 items. The `flatMapValues` function applies a function to each of the element values. The element key will remain unchanged.


``mapValues()`` will not **flatten** the results. Hence, 3 input will have 3 output.
In contrast, ``flatMapValues()`` flatten the output, so 3 input will have 1 output.

In [ ]:
mtcars_mapvalues = mtcarsRDD.mapValues(lambda x: [x, x * 10])
print('MapValue: \n', mtcars_mapvalues.take(5), '\n')

mtcars_flatMapValues = mtcarsRDD.flatMapValues(lambda x: [x, x * 10])
print('FlatMapValues: \n', mtcars_flatMapValues.take(5))

MapValue: 
 [('Mazda RX4', [21.0, 210.0]), ('Mazda RX4 Wag', [21.0, 210.0]), ('Datsun 710', [22.8, 228.0]), ('Hornet 4 Drive', [21.4, 214.0]), ('Hornet Sportabout', [18.7, 187.0])] 

FlatMapValues: 
 [('Mazda RX4', 21.0), ('Mazda RX4', 210.0), ('Mazda RX4 Wag', 21.0), ('Mazda RX4 Wag', 210.0), ('Datsun 710', 22.8)]


### 4.3 RDD example: ``countByKey()`` and ``countByValue()``
``countByKey()`` and ``countByValue()`` count the frequency of each **key** and **value** (resp.),
and return the count as **defaultdict** .

In [ ]:
# dataframe: car brand, new product release year
carData = ss.createDataFrame([
    ('Mazda', 2018),
    ('Mazda', 2019),
    ('Ferrari', 2019),
    ('Porsche', 2019)],
    ["brand", "year"]).rdd

# Get frequency of key (i.e., car brand)
print("Key Frequency:", carData.countByKey())

# Get frequency of value (i.e., how many car released in each year)
print("Value Frequency:", carData.countByValue())

Key Frequency: defaultdict(<class 'int'>, {'Mazda': 2, 'Ferrari': 1, 'Porsche': 1})
Value Frequency: defaultdict(<class 'int'>, {Row(brand='Mazda', year=2018): 1, Row(brand='Mazda', year=2019): 1, Row(brand='Ferrari', year=2019): 1, Row(brand='Porsche', year=2019): 1})


### 4.4 RDD example: A combined used of ``countByKey()``, ``countByValue()`` and ``flatMapValues()`` for car brand count

In [ ]:
# A function to extract brand from car list
def text_processing(text):
  return [w for w in text.split()]

# dataframe: car brand, new product release year
carData = ss.createDataFrame([
    (2018, 'Mazda Ferrari'),
    (2019, 'Porsche Ford Mazda'),
    (2020, 'Ferrari Honda Tesla Mazda')],
    ["year", "brand"]).rdd

extracted_car = carData.map(lambda x: (x.year, text_processing(x.brand)))

# Count No. of brand in each year (e.g., (2018, Mazda): 1)
brand_frequency = extracted_car.flatMapValues(lambda x: x).countByValue()
print(brand_frequency.items())

# Count No. of year containing a particular car brand (e.g., Mazda: 3, Porsche: 1)
year_frequency = extracted_car.flatMapValues(lambda x: x).distinct()\
                        .map(lambda x: (x[1], x[0])).countByKey()
print(year_frequency.items())

# No. of year
number_of_years = extracted_car.count()
print("number_of_year:", number_of_years)

dict_items([((2018, 'Mazda'), 1), ((2018, 'Ferrari'), 1), ((2019, 'Porsche'), 1), ((2019, 'Ford'), 1), ((2019, 'Mazda'), 1), ((2020, 'Ferrari'), 1), ((2020, 'Honda'), 1), ((2020, 'Tesla'), 1), ((2020, 'Mazda'), 1)])
dict_items([('Mazda', 3), ('Ferrari', 2), ('Porsche', 1), ('Ford', 1), ('Honda', 1), ('Tesla', 1)])
number_of_year: 3


### 4.5 Merge and split columns

Sometimes we need to merge multiple columns in a Dataframe into one column, or split a column into multiple columns.
We can easily achieve this by converting a DataFrame to RDD, applying ``map()`` functions to manipulate elements, and then converting the RDD back to a DataFrame.

### 4.5.1 Merge multiple columns
We convert DataFrame to RDD and then apply the map function to merge values and convert elements to Row objects.

In [ ]:
from pyspark.sql import Row

# read csv as dataframe
mtcars = ss.read.csv(path='mtcars.csv',
                        sep=',',
                        encoding='UTF-8',
                        comment=None,
                        header=True,
                        inferSchema=True)

# convert to a key-value pair, merge all columns expcept the 1st one
mtcars_rdd = mtcars.rdd.map(lambda x: Row(model=x[0], values=x[1:]))  # values merged the 2nd to end columns
mtcars_rdd.take(5)


[Row(model='Mazda RX4', values=(21.0, 6, 160.0, 110, 3.9, 2.62, 16.46, 0, 1, 4, 4)),
 Row(model='Mazda RX4 Wag', values=(21.0, 6, 160.0, 110, 3.9, 2.875, 17.02, 0, 1, 4, 4)),
 Row(model='Datsun 710', values=(22.8, 4, 108.0, 93, 3.85, 2.32, 18.61, 1, 1, 4, 1)),
 Row(model='Hornet 4 Drive', values=(21.4, 6, 258.0, 110, 3.08, 3.215, 19.44, 1, 0, 3, 1)),
 Row(model='Hornet Sportabout', values=(18.7, 8, 360.0, 175, 3.15, 3.44, 17.02, 0, 0, 3, 2))]

### 4.5.2 Split multiple columns
We use the above DataFrame as our example data. Again, we need to convert the DataFrame to an RDD to achieve our goal.

Let's split the **values** column into two columns: x1 and x2. The first 4 values will be in column **x1** and the remaining values will be in column **x2**.

In [ ]:
# create a RDD_df with key=col_1, x1=col_2-4 and x2=col_5
mtcars_rdd_2 = mtcars.rdd.map(lambda x: Row(model=x[0], x1=x[1:5], x2=x[5:]))
mtcars_rdd_2.take(5)

# convert RDD back to DataFrame
mtcars_df_2 = ss.createDataFrame(mtcars_rdd_2)
mtcars_df_2.show(5, truncate=False)

+-----------------+---------------------+--------------------------------+
|model            |x1                   |x2                              |
+-----------------+---------------------+--------------------------------+
|Mazda RX4        |{21.0, 6, 160.0, 110}|{3.9, 2.62, 16.46, 0, 1, 4, 4}  |
|Mazda RX4 Wag    |{21.0, 6, 160.0, 110}|{3.9, 2.875, 17.02, 0, 1, 4, 4} |
|Datsun 710       |{22.8, 4, 108.0, 93} |{3.85, 2.32, 18.61, 1, 1, 4, 1} |
|Hornet 4 Drive   |{21.4, 6, 258.0, 110}|{3.08, 3.215, 19.44, 1, 0, 3, 1}|
|Hornet Sportabout|{18.7, 8, 360.0, 175}|{3.15, 3.44, 17.02, 0, 0, 3, 2} |
+-----------------+---------------------+--------------------------------+
only showing top 5 rows



## 5. TODO Exercises

<font color='blue'>TODO 1: In the previous tutorial, we extracted continuous patterns of size k from a DNA string. Now, suppose we have a DNA sequence that appears more than 10 times, as provided in the cell below. Please convert this data into a DataFrame with two columns: (sequence, count).</font>

In [ ]:
# TODO 1: Transform the provided data into a DataFrame with two columns: (sequence, count).

from operator import add
from pyspark.sql import Row

data = [
    ("ctaag", 103),
    ("gccta", 96),
    ("cctaa", 95),
    ("aagcc", 94),
    ("agcct", 94),
    ("taagc", 93),
    ("ttttt", 30),
    ("aaaaa", 20),
    ("aaaat", 20),
    ("atttt", 18),
    ("aaatt", 16),
    ("actaa", 15),
    ("ttttc", 14),
    ("taaga", 13),
    ("aattt", 13),
    ("tttta", 12),
    ("gtttt", 11)
]

rdd = ...

+--------+-----+
|sequence|count|
+--------+-----+
|   ctaag|  103|
|   gccta|   96|
|   cctaa|   95|
|   aagcc|   94|
|   agcct|   94|
|   taagc|   93|
|   ttttt|   30|
|   aaaaa|   20|
|   aaaat|   20|
|   atttt|   18|
|   aaatt|   16|
|   actaa|   15|
|   ttttc|   14|
|   taaga|   13|
|   aattt|   13|
|   tttta|   12|
|   gtttt|   11|
+--------+-----+



<font color='blue'>TODO 2: Given two datasets hills2000.txt and hills.txt, they are available in blackboard, please download them and upload to Colab or placing in the approporiate directory for loading. </font>

<font color='blue'>Calculate the average speed (in miles per hour) for each race in both datasets, combining them into a single DataFrame, and identify the top 5 races with the highest average speed.</font>

<font color='blue'>For races in hills2000.txt that have both time (male) and timef (female) columns, use the average of the two times to compute speed.</font>

<font color='blue'>Please note that in this question, we may need to use the ``col`` SQL function.

You can import it using ``from pyspark.sql.functions import col``</font>

<font color='blue'>Reference: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.col.html</font>


<font color="blue">TODO 2.1: Load the dataset from ``hills2000.txt`` and create the RDDs for races that contain both the male time and female time columns. Then, compute the average time using the formula ``(time+timef)/2`` and store the result in a new column called avg_speed.</font>



In [ ]:
from pyspark.sql.functions import col
hills2000_df = ...

+--------------------+----+-----+-----------------+-----------------+-------------------+
|            rownames|dist|climb|             time|            timef|           avg_time|
+--------------------+----+-----+-----------------+-----------------+-------------------+
|       Tiso Carnethy| 6.0| 2500|0.782222222222222|0.919166666666667| 0.8506944444444445|
|             Criffel| 7.0| 1800|0.793333333333333| 1.00333333333333| 0.8983333333333315|
|          Chapelgill| 1.5| 1400|0.314444444444444|0.376666666666667| 0.3455555555555555|
|        Norman's Law| 5.0|  700|0.464166666666667|0.609166666666667|  0.536666666666667|
|        Craig Dunain| 6.0|  900|0.546111111111111|0.625833333333333|  0.585972222222222|
|         Knockfarrel| 5.0| 1200|0.623333333333333|0.734166666666667|            0.67875|
|              Screel| 4.0| 1300|0.458888888888889|0.543611111111111|            0.50125|
|   O.P.S. Clachnaben|10.5| 3500| 1.27888888888889| 1.48777777777778|  1.383333333333335|
|       Wh

<font color='blue'>TODO 2.2: Once we have calculated the average time from Task 2.1, we can determine the average speed by dividing distance by the ``avg_time``. Store the resulting value in a new column named ``avg_speed``.</font>

In [ ]:
hills2000_df = ...

+--------------------+----+-----+-----------------+-----------------+-------------------+------------------+
|            rownames|dist|climb|             time|            timef|           avg_time|             speed|
+--------------------+----+-----+-----------------+-----------------+-------------------+------------------+
|       Tiso Carnethy| 6.0| 2500|0.782222222222222|0.919166666666667| 0.8506944444444445| 7.053061224489795|
|             Criffel| 7.0| 1800|0.793333333333333| 1.00333333333333| 0.8983333333333315| 7.792207792207808|
|          Chapelgill| 1.5| 1400|0.314444444444444|0.376666666666667| 0.3455555555555555| 4.340836012861737|
|        Norman's Law| 5.0|  700|0.464166666666667|0.609166666666667|  0.536666666666667| 9.316770186335399|
|        Craig Dunain| 6.0|  900|0.546111111111111|0.625833333333333|  0.585972222222222|10.239393221142455|
|         Knockfarrel| 5.0| 1200|0.623333333333333|0.734166666666667|            0.67875| 7.366482504604052|
|              Scre

<font color='blue'>TODO 2.3: Perform the same calculation on another dataset, ``hills.txt``, by dividing the distance by the time to obtain the speed.</font>

In [ ]:
hills_df = ...

+------------+----+-----+-----------------+------------------+
|    rownames|dist|climb|             time|             speed|
+------------+----+-----+-----------------+------------------+
| Greenmantle| 2.4|  650|0.268055555555556| 8.953367875647654|
|    Carnethy| 6.0| 2500|0.805833333333333| 7.445708376421927|
|Craig Dunain| 6.0|  900|0.560833333333333|10.698365527488862|
|     Ben Rha| 7.5|  800|             0.76| 9.868421052631579|
|  Ben Lomond| 8.0| 3070| 1.03777777777778|7.7087794432548025|
|    Goatfell| 8.0| 2866| 1.22027777777778| 6.555884361484168|
|Bens of Jura|16.0| 7500| 3.41027777777778| 4.691699926692185|
| Cairnpapple| 6.0|  800|0.606111111111111| 9.899175068744272|
|      Scolty| 5.0|  800|0.495833333333333|10.084033613445385|
|    Traprain| 6.0|  650|           0.6625| 9.056603773584905|
| Lairig Ghru|28.0| 2100| 3.21111111111111| 8.719723183391007|
|      Dollar| 5.0| 2000|           0.7175| 6.968641114982578|
|     Lomonds| 9.5| 2200| 1.08333333333333| 8.769230769

<font color='blue'>TODO 2.4: Merge both dataframes into one, ensuring it contains the columns (``rownames``, ``climb``, ``dist``, and ``avg_speed``).</font>

In [ ]:
hills_speed = ...

+------------+-----+----+------------------+
|    rownames|climb|dist|             speed|
+------------+-----+----+------------------+
| Greenmantle|  650| 2.4| 8.953367875647654|
|    Carnethy| 2500| 6.0| 7.445708376421927|
|Craig Dunain|  900| 6.0|10.698365527488862|
|     Ben Rha|  800| 7.5| 9.868421052631579|
|  Ben Lomond| 3070| 8.0|7.7087794432548025|
|    Goatfell| 2866| 8.0| 6.555884361484168|
|Bens of Jura| 7500|16.0| 4.691699926692185|
| Cairnpapple|  800| 6.0| 9.899175068744272|
|      Scolty|  800| 5.0|10.084033613445385|
|    Traprain|  650| 6.0| 9.056603773584905|
| Lairig Ghru| 2100|28.0| 8.719723183391007|
|      Dollar| 2000| 5.0| 6.968641114982578|
|     Lomonds| 2200| 9.5| 8.769230769230797|
| Cairn Table|  500| 6.0| 8.157099697885192|
|  Eildon Two| 1500| 4.5|10.024752475247523|
|   Cairngorm| 3000|10.0| 8.304498269896172|
| Seven Hills| 2200|14.0| 8.535139712108371|
|  Knock Hill|  350| 3.0| 2.288620470438658|
|  Black Hill| 1000| 4.5|15.502392344497595|
|  Creag B

<font color='blue'>TODO 2.5: Retrieve the top 5 races with the highest average speed, sorted in descending order. If there are multiple entries for the same rowname, we should calculate the average speed.</font>

In [ ]:
...

+------------+------------------+
|    rownames|        avg(speed)|
+------------+------------------+
|      Acmony|14.319809069212397|
|  Black Hill|11.832828825310022|
|Kildcon Hill|11.285266457680265|
|  Caerketton|10.982306284319701|
|   Largo Law|10.501750291715288|
+------------+------------------+
only showing top 5 rows



<font color='blue'>TODO 2.6: Next, we will calculate the difficulty score for each race by dividing the ``climb`` by the ``distance``. The results will be stored in a new column named ``diff_score``.</font>

In [ ]:
hills_union_diff = ...


+------------+-----+----+------------------+------------------+
|    rownames|climb|dist|             speed|        diff_score|
+------------+-----+----+------------------+------------------+
| Greenmantle|  650| 2.4| 8.953367875647654|270.83333333333337|
|    Carnethy| 2500| 6.0| 7.445708376421927| 416.6666666666667|
|Craig Dunain|  900| 6.0|10.698365527488862|             150.0|
|     Ben Rha|  800| 7.5| 9.868421052631579|106.66666666666667|
|  Ben Lomond| 3070| 8.0|7.7087794432548025|            383.75|
|    Goatfell| 2866| 8.0| 6.555884361484168|            358.25|
|Bens of Jura| 7500|16.0| 4.691699926692185|            468.75|
| Cairnpapple|  800| 6.0| 9.899175068744272|133.33333333333334|
|      Scolty|  800| 5.0|10.084033613445385|             160.0|
|    Traprain|  650| 6.0| 9.056603773584905|108.33333333333333|
| Lairig Ghru| 2100|28.0| 8.719723183391007|              75.0|
|      Dollar| 2000| 5.0| 6.968641114982578|             400.0|
|     Lomonds| 2200| 9.5| 8.769230769230

<font color="blue">TODO 2.7: Next, we need to categorize the difficulty levels as 'Easy', 'Moderate', or 'Difficult'. If the diff_score is less than 200, it will be classified as 'Easy'; if it's less than 350, it will be classified as 'Moderate'; otherwise, it will be classified as 'Difficult'. For this task, you may need to use the when function.</font>

<font color="blue">The when() function takes a condition and a value: if the condition evaluates to True, the specified value is returned. You can chain multiple when() calls to handle several conditions sequentially.</font>

<font color="blue">Ref: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html
</font>


In [ ]:
from pyspark.sql.functions import when
hills_union_diff_level = ...

+------------+-----+----+------------------+------------------+---------+
|    rownames|climb|dist|             speed|        diff_score|    level|
+------------+-----+----+------------------+------------------+---------+
| Greenmantle|  650| 2.4| 8.953367875647654|270.83333333333337| Moderate|
|    Carnethy| 2500| 6.0| 7.445708376421927| 416.6666666666667|Difficult|
|Craig Dunain|  900| 6.0|10.698365527488862|             150.0|     Easy|
|     Ben Rha|  800| 7.5| 9.868421052631579|106.66666666666667|     Easy|
|  Ben Lomond| 3070| 8.0|7.7087794432548025|            383.75|Difficult|
|    Goatfell| 2866| 8.0| 6.555884361484168|            358.25|Difficult|
|Bens of Jura| 7500|16.0| 4.691699926692185|            468.75|Difficult|
| Cairnpapple|  800| 6.0| 9.899175068744272|133.33333333333334|     Easy|
|      Scolty|  800| 5.0|10.084033613445385|             160.0|     Easy|
|    Traprain|  650| 6.0| 9.056603773584905|108.33333333333333|     Easy|
| Lairig Ghru| 2100|28.0| 8.7197231833

<font color="blue">TODO 2.8: Finally, displaying the count of races in each difficulty category.
</font>


In [ ]:
...

+---------+-----+
|    level|count|
+---------+-----+
|Difficult|   29|
|     Easy|   29|
| Moderate|   33|
+---------+-----+



<font color='#0000FF'> TODO 3: Write a module to read setenceData as RDD, then split the text by space and lowercase the word. Finally, do the following counts:
* **No. of Document**: Number of document
* **Term_Frequency**: Words count of each document in any output format. E.g., ((Doc ID, word), count)((0, 'studying'), 2), (1, 'studying;), 1)
* **Document_Frequency**: No. of document containing a word. E.g., (Word, Count, ('studying', 2)</font>

<font color='#0000FF'> **Hints**: You may need to use the following functions (also look into Example 3.4.5):

* `lower()`
* `split()`
* `flatMapValues()`
* `distinct()`
* `count()`
* `countByKey()`
* `countByValue()`
</font>

In [ ]:
#TODO 3. Write a module to read setenceData as RDD, then split the text by space and lowercase the word.
#        Finally, do the following counts:
#        No. of Document: Number of document
#        Term_Frequency: Words count of each document in any output format. E.g., ((Doc ID, word), count)((0, 'studying'), 2), (1, 'studying;), 1)
#.       Document_Frequency: No. of document containing a word. E.g., (Word, Count, ('studying', 2)
sentenceData = ss.createDataFrame([
    (0, "I am Billy studying in Computer and I am also studying in Business Course"),
    (1, "Ada is studying in Computer"),
    (2, "Billy and Ada know each other in a computer course course")],
    ["document", "sentence"])

## TODO 3.1 print out the number of documents.



## TODO 3.2 count words in each document



## TODO 3.3 No of documents containing a term


Number of documents: 3

Term_Frequency:
i: 2 in D0
am: 2 in D0
billy: 1 in D0
studying: 2 in D0
in: 2 in D0
computer: 1 in D0
and: 1 in D0
also: 1 in D0
business: 1 in D0
course: 1 in D0
ada: 1 in D1
is: 1 in D1
studying: 1 in D1
in: 1 in D1
computer: 1 in D1
billy: 1 in D2
and: 1 in D2
ada: 1 in D2
know: 1 in D2
each: 1 in D2
other: 1 in D2
in: 1 in D2
a: 1 in D2
computer: 1 in D2
course: 2 in D2

Document Frequency:
i:1|am:1|billy:2|studying:2|in:3|computer:3|and:2|also:1|business:1|course:2|ada:2|is:1|know:1|each:1|other:1|a:1|